# Chapter 6 — Neural Networks Learn Correlation

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 6:** Neural Networks Learn Correlation

## Isi Chapter 6

1. Streetlight problem: data yang harus dipelajari jaringan
2. Matriks dan hubungannya dengan data
3. Membangun neural network untuk streetlight
4. Full, batch, dan stochastic gradient descent
5. Neural network belajar korelasi
6. Overfitting
7. Conflicting pressure dan indirect correlation
8. Backpropagation: atribusi error jarak jauh
9. Linear vs nonlinear (relu)
10. Deep network pertama dan backpropagation dalam kode

## 1. Streetlight problem dan matriks

Masalah streetlight: di persimpangan dengan lampu yang tidak dikenal, kapan aman
menyeberang? Saya mengamati enam kondisi lampu (kiri, tengah, kanan) dan melihat apakah
orang berjalan (`WALK`) atau berhenti (`STOP`). Pola yang muncul: lampu **tengah**
berkorelasi sempurna dengan keputusan untuk berjalan.

Data ini dibagi menjadi dua dataset: kondisi lampu (yang diketahui) dan keputusan
berjalan/berhenti (yang ingin diketahui). Keduanya diubah menjadi matriks berisi 0 dan 1.
Setiap kolom mewakili satu lampu, dan setiap baris mewakili satu contoh pengamatan.

In [1]:
import numpy as np

# Input: kondisi 3 lampu (kiri, tengah, kanan) untuk 6 pengamatan
streetlights = np.array([[1, 0, 1],
                         [0, 1, 1],
                         [0, 0, 1],
                         [1, 1, 1],
                         [0, 1, 1],
                         [1, 0, 1]])

# Output: 0 = STOP, 1 = WALK
walk_vs_stop = np.array([[0],
                         [1],
                         [0],
                         [1],
                         [1],
                         [0]])

print('streetlights shape :', streetlights.shape)
print('walk_vs_stop shape :', walk_vs_stop.shape)
print(streetlights)

streetlights shape : (6, 3)
walk_vs_stop shape : (6, 1)
[[1 0 1]
 [0 1 1]
 [0 0 1]
 [1 1 1]
 [0 1 1]
 [1 0 1]]


Saya perhatikan bahwa matriks ini hanya mewakili pola yang sama dengan matriks lain
yang dikalikan skalar (misalnya nilai 0 sampai 1 atau 0 sampai 20). Pola itulah yang
penting, bukan angka pastinya. Representasi 0/1 ini juga *lossless*: saya bisa
mengubah kembali ke STOP/WALK tanpa kehilangan informasi.

## 2. Operasi elementwise di NumPy

Sebelum membangun jaringan, saya perlu mengenal dua operasi NumPy yang sering dipakai:
perkalian dan penjumlahan elementwise, serta operasi dengan skalar.

In [2]:
a = np.array([0, 1, 2, 1])
b = np.array([2, 2, 2, 3])

print('a * b   =', a * b)     # elementwise multiplication
print('a + b   =', a + b)     # elementwise addition
print('a * 0.5 =', a * 0.5)   # vector-scalar multiplication
print('a + 0.5 =', a + 0.5)   # vector-scalar addition

a * b   = [0 2 4 3]
a + b   = [2 3 4 4]
a * 0.5 = [0.  0.5 1.  0.5]
a + 0.5 = [0.5 1.5 2.5 1.5]


## 3. Melatih seluruh dataset

Neural network sebelumnya hanya belajar dari satu contoh. Di sini saya melatih jaringan
pada keenam streetlight sekaligus. Setiap contoh diprediksi dan bobotnya diperbarui
satu per satu, lalu proses ini diulang untuk 40 epoch. Ini disebut **stochastic gradient
descent**: bobot diperbarui setiap satu contoh.

In [3]:
weights = np.array([0.5, 0.48, -0.7])
alpha = 0.1

streetlights = np.array([[1, 0, 1],
                         [0, 1, 1],
                         [0, 0, 1],
                         [1, 1, 1],
                         [0, 1, 1],
                         [1, 0, 1]])
walk_vs_stop = np.array([0, 1, 0, 1, 1, 0])

for iteration in range(40):
    error_for_all_lights = 0
    for row_index in range(len(walk_vs_stop)):
        input = streetlights[row_index]
        goal_prediction = walk_vs_stop[row_index]

        prediction = input.dot(weights)
        error = (goal_prediction - prediction) ** 2
        error_for_all_lights += error

        delta = prediction - goal_prediction
        weights = weights - (alpha * (input * delta))

    if iteration + 1 in (1, 2, 3, 4, 5, 10, 20, 30, 40):
        print('Epoch', iteration + 1, '| error', round(float(error_for_all_lights), 6))

print('\nBobot akhir:', np.round(weights, 4))

Epoch 1 | error 2.656123
Epoch 2 | error 0.96287
Epoch 3 | error 0.550917
Epoch 4 | error 0.364458
Epoch 5 | error 0.251677
Epoch 10 | error 0.055649
Epoch 20 | error 0.009117
Epoch 30 | error 0.002181
Epoch 40 | error 0.000534

Bobot akhir: [ 0.0139  1.0138 -0.016 ]


Error turun dari sekitar `2.66` pada epoch pertama menjadi sekitar `0.0005` pada epoch
ke-40. Bobot akhirnya sekitar `[0.01, 1.01, -0.02]`: bobot lampu tengah mendekati 1,
sedangkan bobot lampu kiri dan kanan mendekati 0.

## 4. Full, batch, dan stochastic gradient descent

Ketiga metode ini berbeda dalam kapan bobot diperbarui:

- **Stochastic**: bobot diperbarui setelah setiap contoh. Ini yang dipakai pada kode di atas.
- **Full (average) gradient descent**: bobot diperbarui sekali setelah seluruh dataset dilihat, dengan rata-rata `weight_delta`.
- **Batch gradient descent**: bobot diperbarui setelah setiap batch berisi beberapa contoh. Ukuran batch biasanya antara 8 sampai 256.

Bagian ini hanya penjelasan konsep, jadi tidak ada kode tambahan.

## 5. Neural network belajar korelasi

Jaringan sebenarnya tidak "membaca" lampu lalu lintas. Yang dilakukan jaringan adalah
mencari korelasi antara input dan output. Bobot yang tinggi berarti input tersebut
berkorelasi dengan output. Pada hasil di atas, bobot lampu tengah tinggi, sehingga
lampu tengah adalah input yang berkorelasi dengan keputusan berjalan.

Setiap contoh memberikan "tekanan" pada bobot. Dari tabel tekanan yang saya tulis
dengan tangan, bobot lampu tengah selalu mendapat tekanan naik ketika lampu tengah
menyala, sedangkan bobot kiri dan kanan mendapat tekanan turun karena tidak berkorelasi.
Ketiga bobot memakai error yang sama, lalu setiap bobot menerima koreksi sebesar
delta dikali inputnya sendiri. Jika inputnya 0, bobot itu tidak ikut berubah.

## 6. Overfitting

Overfitting terjadi ketika bobot secara kebetulan membuat prediksi benar untuk beberapa
contoh, tetapi tidak menangkap korelasi yang sebenarnya. Contoh: bobot kiri `0.5` dan
bobot kanan `-0.5` memberi prediksi `0` untuk contoh pertama `[1, 0, 1]`. Error-nya nol,
tetapi bobot itu tidak akan benar untuk streetlight lain.

In [4]:
# Overfitting: bobot yang kebetulan benar untuk contoh pertama
overfit_weights = np.array([0.5, 0.0, -0.5])
contoh_pertama = streetlights[0]           # [1, 0, 1]
prediksi = contoh_pertama.dot(overfit_weights)
print('prediksi contoh pertama :', prediksi)
print('target                  :', walk_vs_stop[0])
print('error                   :', (walk_vs_stop[0] - prediksi) ** 2)

prediksi contoh pertama : 0.0
target                  : 0
error                   : 0.0


Pelajaran yang saya catat: karena error dibagi ke semua bobot, satu contoh saja tidak
cukup untuk menghindari overfitting. Contoh-contoh lain dalam dataset memaksa bobot
untuk tetap mencari korelasi yang benar. Tantangan terbesar dalam deep learning adalah
membuat jaringan **generalize**, bukan hanya **memorize**.

## 7. Conflicting pressure dan indirect correlation

Pada beberapa kasus, sebuah bobot mendapat tekanan naik dan turun dalam jumlah yang sama.
Bobot seperti ini tidak membantu prediksi, sehingga jaringan biasanya akan mendorongnya
mendekati 0. Ada pula dataset yang tidak memiliki korelasi sama sekali antara input dan
output. Untuk kasus itu, jaringan perlu **dua lapisan**: lapisan pertama membuat data
perantara yang berkorelasi dengan output, lalu lapisan kedua memakai data perantara itu
untuk prediksi.

Dengan kata lain, tumpukan dua neural network dapat dianggap sebagai satu jaringan
dengan lapisan tersembunyi (*hidden layer*) `layer_1`.

## 8. Backpropagation: atribusi error jarak jauh

Pada jaringan dua lapis, delta di output dikali bobot yang menghubungkannya ke lapisan
tersembunyi memberi delta untuk setiap node di `layer_1`. Bobot yang lebih besar
berarti node tersebut berkontribusi lebih besar pada error. Proses ini disebut
**backpropagation**: sinyal delta dipindahkan mundur melalui bobot.

Saya memahami bahwa `layer_1` tidak perlu tahu apa yang dilakukan `layer_2` secara
langsung. Ia cukup menerima delta yang sudah dikalikan bobot.

## 9. Linear vs nonlinear

Dua perkalian berurutan tanpa fungsi tambahan hanya menghasilkan satu transformasi linear.
Karena itu, tiga lapis tanpa nonlinearitas tidak lebih kuat daripada dua lapis.
Solusinya adalah **relu**: jika nilai node negatif, node tersebut diatur menjadi 0.
Dengan cara ini, node bisa berkorelasi dengan input secara kondisional.

Turunan relu (`relu2deriv`) bernilai 1 jika output lebih besar dari 0, dan 0 jika tidak.
Saya cek kedua fungsi ini dengan contoh kecil:

In [5]:
def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0

contoh = np.array([-1.0, 0.0, 2.0])
print('relu(contoh)       :', relu(contoh))
print('relu2deriv(contoh) :', relu2deriv(relu(contoh)))

relu(contoh)       : [-0.  0.  2.]
relu2deriv(contoh) : [False False  True]


## 10. Deep network pertama: forward propagation

Sekarang saya membangun jaringan tiga lapis dengan `hidden_size = 4`. Bobotnya diinisialisasi
secara acak dengan `np.random.seed(1)` agar hasilnya bisa diulang. Prediksi dihitung dengan
urutan: input -> `layer_1` (dengan relu) -> `layer_2`.

In [6]:
np.random.seed(1)

streetlights = np.array([[1, 0, 1],
                         [0, 1, 1],
                         [0, 0, 1],
                         [1, 1, 1]])
walk_vs_stop = np.array([[1, 1, 0, 0]]).T

alpha = 0.2
hidden_size = 4

weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

layer_0 = streetlights[0:1]
layer_1 = relu(np.dot(layer_0, weights_0_1))
layer_2 = np.dot(layer_1, weights_1_2)

print('layer_0 :', layer_0)
print('layer_1 :', np.round(layer_1, 4))
print('layer_2 :', np.round(layer_2, 4))

layer_0 : [[1 0 1]]
layer_1 : [[-0.      0.5183 -0.     -0.    ]]
layer_2 : [[0.3919]]


## 11. Backpropagation dalam kode

Kode pembelajaran di bawah ini sama seperti prediksi di atas, ditambah langkah
backpropagation:

1. `layer_2_delta` dihitung dengan cara yang sama seperti jaringan dua lapis.
2. `layer_1_delta` dihitung dengan mengalikan `layer_2_delta` dengan `weights_1_2.T`,
   lalu dikalikan `relu2deriv(layer_1)`. Perkalian terakhir ini mematikan node yang
   tidak ikut dalam prediksi, karena node tersebut tidak mungkin berkontribusi pada error.
3. Setiap bobot diperbarui dengan `input.T.dot(delta)` dikali `alpha`.

Pada setiap iterasi, seluruh contoh diproses satu per satu, lalu error dijumlahkan.

In [7]:
np.random.seed(1)

def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0

streetlights = np.array([[1, 0, 1],
                         [0, 1, 1],
                         [0, 0, 1],
                         [1, 1, 1]])
walk_vs_stop = np.array([[1, 1, 0, 0]]).T

alpha = 0.2
hidden_size = 4

weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

for iteration in range(60):
    layer_2_error = 0
    for i in range(len(streetlights)):
        layer_0 = streetlights[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)

        layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i+1]) ** 2)

        layer_2_delta = (walk_vs_stop[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if iteration % 10 == 9:
        print('Error:', layer_2_error)

Error: 0.6342311598444467
Error: 0.35838407676317513
Error: 0.0830183113303298
Error: 0.006467054957103705
Error: 0.0003292669000750734
Error: 1.5055622665134859e-05


Error turun dari `0.634` pada iterasi ke-10 menjadi sekitar `1.5e-05` pada iterasi ke-60.
Jaringan tiga lapis ini sekarang bisa memprediksi keempat streetlight dengan benar.

## Mengapa deep network penting

Dalam dataset gambar kucing, tidak ada satu piksel pun yang berkorelasi langsung dengan
keberadaan kucing. Yang berkorelasi adalah **konfigurasi** piksel, misalnya bentuk telinga
atau mata. Lapisan tersembunyi mempelajari konfigurasi-konfigurasi ini, lalu lapisan
terakhir memakai kumpulan konfigurasi itu untuk prediksi. Inilah inti deep learning:
setiap node di lapisan tengah mewakili ada atau tidaknya sebuah pola input.

## Key Ideas — Chapter 6

- Dataset dipetakan ke matriks: setiap baris adalah contoh, setiap kolom adalah fitur.
- Stochastic gradient descent memperbarui bobot per contoh; full dan batch gradient descent memperbaruinya per kelompok.
- Neural network mencari korelasi antara input dan output. Bobot tinggi berarti korelasi kuat.
- Overfitting terjadi ketika error nol dicapai tanpa menangkap korelasi yang benar.
- Backpropagation memindahkan delta dari output ke lapisan tersembunyi melalui bobot.
- Relu memberi nonlinearitas sehingga tumpukan lapisan memiliki kekuatan lebih dari satu lapis.

## Kesimpulan

Chapter 6 menjelaskan bagaimana neural network belajar korelasi dari dataset, dan
bagaimana backpropagation memungkinkan jaringan tiga lapis belajar dari data yang tidak
memiliki korelasi langsung antara input dan output. Chapter berikutnya membahas
cara membuat korelasi yang lebih kompleks dengan lebih banyak data.